# Supplementary — Institutional Landscape

Which institutions produce global health research, how do they specialize, and where do collaborations cluster? This notebook is especially relevant for students and early-career researchers looking to identify institutions active in specific topics.

### Analyses
1. **Top institutions by paper count** — who produces the most global health research?
2. **Institution × topic heatmap** — which institutions specialize in which topics?
3. **Institution country distribution** — geographic concentration of research production
4. **Collaboration density by topic** — how many institutions per paper, by topic?
5. **Rising institutions** — which institutions are growing their output fastest?

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)

DB = 'data/global_health.duckdb'
con = duckdb.connect(DB, read_only=True)

## Load core data

In [ ]:
# Topic taxonomy for labels
topic_labels = pd.read_csv('data/taxonomy/topic_taxonomy.csv')
cat_names = topic_labels.drop_duplicates('category_letter')[['category_letter', 'category_name']]
cat_map = dict(zip(cat_names['category_letter'], cat_names['category_name']))

# All authorships joined with work metadata
inst_data = con.execute("""
    SELECT a.openalex_id, a.author_name, a.position,
           a.institution_name, a.institution_country,
           w.publication_year, w.topic_category, w.topic_subtopic,
           w.method_type
    FROM authorships a
    JOIN works w ON a.openalex_id = w.openalex_id
    WHERE a.institution_name IS NOT NULL
      AND a.institution_name != ''
      AND w.topic_category IS NOT NULL
""").fetchdf()

inst_data['topic_name'] = inst_data['topic_category'].map(cat_map)

print(f'Authorship records with institution: {len(inst_data):,}')
print(f'Unique institutions: {inst_data["institution_name"].nunique():,}')
print(f'Unique papers: {inst_data["openalex_id"].nunique():,}')

---
## S1. Top Institutions by Paper Count

Which institutions appear most frequently in global health research? Counted by unique papers (an institution with 3 authors on one paper counts once).

In [ ]:
# Count unique papers per institution
inst_paper_counts = (
    inst_data
    .groupby(['institution_name', 'institution_country'])
    ['openalex_id'].nunique()
    .reset_index(name='n_papers')
    .sort_values('n_papers', ascending=False)
)

top_25 = inst_paper_counts.head(25)

fig, ax = plt.subplots(figsize=(12, 10))
labels = [f"{row['institution_name']} ({row['institution_country']})"
          for _, row in top_25.iterrows()]
ax.barh(range(len(top_25)), top_25['n_papers'], color='#2171b5')
ax.set_yticks(range(len(top_25)))
ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Number of papers')
ax.set_title('Top 25 Institutions in Global Health Research (by unique paper count)')

for i, (_, row) in enumerate(top_25.iterrows()):
    ax.text(row['n_papers'] + 0.5, i, str(row['n_papers']), va='center', fontsize=9)

fig.tight_layout()
plt.show()

print(f'\nTotal institutions with papers: {len(inst_paper_counts):,}')
print(f'Institutions with ≥10 papers: {(inst_paper_counts["n_papers"] >= 10).sum()}')
print(f'Institutions with only 1 paper: {(inst_paper_counts["n_papers"] == 1).sum()}')

---
## S2. Institution × Topic Heatmap

What does each major institution specialize in? This heatmap shows the topic profile (row-normalized) for the top 20 institutions — useful for students choosing where to study a specific area.

In [ ]:
top_20_insts = inst_paper_counts.head(20)['institution_name'].tolist()

# Cross-tab: institution × topic
inst_topic = (
    inst_data[inst_data['institution_name'].isin(top_20_insts)]
    .groupby(['institution_name', 'topic_name'])
    ['openalex_id'].nunique()
    .reset_index(name='n_papers')
    .pivot_table(index='institution_name', columns='topic_name',
                 values='n_papers', fill_value=0)
)

# Row-normalize to show each institution's topic portfolio
inst_topic_prop = inst_topic.div(inst_topic.sum(axis=1), axis=0)

# Sort institutions by total papers
inst_order = inst_paper_counts[inst_paper_counts['institution_name'].isin(top_20_insts)]
inst_topic_prop = inst_topic_prop.loc[
    [n for n in inst_order['institution_name'] if n in inst_topic_prop.index]
]

fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(inst_topic_prop, annot=True, fmt='.0%', cmap='YlOrRd', ax=ax,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'Share of institution papers'})
ax.set_title('Institution Topic Specialization (top 20 institutions, row-normalized)')
ax.set_ylabel('')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
plt.show()

---
## S3. Institution Country Distribution

How geographically concentrated is global health research production? Are LMIC-based institutions well represented?

In [ ]:
# Papers per institution country (counting unique papers)
country_papers = (
    inst_data
    .groupby('institution_country')
    ['openalex_id'].nunique()
    .reset_index(name='n_papers')
    .sort_values('n_papers', ascending=False)
)

# Also count unique institutions per country
country_insts = (
    inst_data
    .groupby('institution_country')
    ['institution_name'].nunique()
    .reset_index(name='n_institutions')
)

country_summary = country_papers.merge(country_insts, on='institution_country')

# Top 25 countries
top_countries = country_summary.head(25)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 10))

# Papers per country
ax1.barh(range(len(top_countries)), top_countries['n_papers'], color='#2171b5')
ax1.set_yticks(range(len(top_countries)))
ax1.set_yticklabels(top_countries['institution_country'])
ax1.invert_yaxis()
ax1.set_xlabel('Number of papers')
ax1.set_title('Papers by Institution Country')

# Institutions per country
ax2.barh(range(len(top_countries)), top_countries['n_institutions'], color='#e6550d')
ax2.set_yticks(range(len(top_countries)))
ax2.set_yticklabels(top_countries['institution_country'])
ax2.invert_yaxis()
ax2.set_xlabel('Number of unique institutions')
ax2.set_title('Unique Institutions by Country')

fig.suptitle('Geographic Distribution of Global Health Research Production', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

# Concentration stats
total_papers = country_summary['n_papers'].sum()
top5_share = country_summary.head(5)['n_papers'].sum() / total_papers
top10_share = country_summary.head(10)['n_papers'].sum() / total_papers
print(f'Top 5 countries produce {top5_share:.0%} of all papers')
print(f'Top 10 countries produce {top10_share:.0%} of all papers')

---
## S4. Collaboration Density by Topic

How many distinct institutions contribute to a typical paper? Higher collaboration density might indicate more complex, multi-site research. Lower density might indicate single-institution studies.

In [ ]:
# Count unique institutions per paper
insts_per_paper = (
    inst_data
    .groupby(['openalex_id', 'topic_name'])
    ['institution_name'].nunique()
    .reset_index(name='n_institutions')
)

# Also count unique countries per paper (international collaboration)
countries_per_paper = (
    inst_data
    .groupby(['openalex_id', 'topic_name'])
    ['institution_country'].nunique()
    .reset_index(name='n_countries')
)

collab = insts_per_paper.merge(countries_per_paper, on=['openalex_id', 'topic_name'])
collab['is_international'] = collab['n_countries'] > 1

# Average collaboration metrics by topic
collab_by_topic = (
    collab
    .groupby('topic_name')
    .agg(
        mean_institutions=('n_institutions', 'mean'),
        mean_countries=('n_countries', 'mean'),
        intl_rate=('is_international', 'mean'),
        n_papers=('openalex_id', 'count'),
    )
    .sort_values('mean_institutions', ascending=True)
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Mean institutions per paper
collab_by_topic['mean_institutions'].plot(kind='barh', ax=ax1, color='#2171b5')
ax1.set_xlabel('Mean institutions per paper')
ax1.set_title('Collaboration Density by Topic')

# International collaboration rate
collab_by_topic_sorted = collab_by_topic.sort_values('intl_rate', ascending=True)
collab_by_topic_sorted['intl_rate'].plot(kind='barh', ax=ax2, color='#e6550d')
ax2.set_xlabel('International collaboration rate')
ax2.set_title('International Collaboration Rate by Topic')

fig.tight_layout()
plt.show()

collab_by_topic.sort_values('mean_institutions', ascending=False)

---
## S5. Rising Institutions

Which institutions are growing their global health research output the fastest? Comparing the most recent period to the earliest helps identify emerging research hubs.

In [ ]:
# Split into early and late periods
years = inst_data['publication_year'].dropna()
mid_year = int(years.median())

early = inst_data[inst_data['publication_year'] <= mid_year]
late = inst_data[inst_data['publication_year'] > mid_year]

early_counts = (
    early.groupby('institution_name')['openalex_id'].nunique()
    .reset_index(name='early_papers')
)
late_counts = (
    late.groupby('institution_name')['openalex_id'].nunique()
    .reset_index(name='late_papers')
)

growth = early_counts.merge(late_counts, on='institution_name', how='outer').fillna(0)
growth['total'] = growth['early_papers'] + growth['late_papers']

# Only consider institutions with at least 5 total papers
growth = growth[growth['total'] >= 5].copy()
growth['growth_rate'] = (growth['late_papers'] - growth['early_papers']) / growth['early_papers'].replace(0, 1)
growth['absolute_growth'] = growth['late_papers'] - growth['early_papers']

# Top risers by absolute growth (more meaningful than % for large institutions)
top_risers = growth.nlargest(15, 'absolute_growth')

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(len(top_risers)), top_risers['absolute_growth'], color='#2ca02c')
ax.set_yticks(range(len(top_risers)))
ax.set_yticklabels(top_risers['institution_name'], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel(f'Growth in papers (post-{mid_year} minus pre-{mid_year})')
ax.set_title('Fastest-Growing Institutions in Global Health Research')

for i, (_, row) in enumerate(top_risers.iterrows()):
    ax.text(row['absolute_growth'] + 0.2, i,
            f"{int(row['early_papers'])} → {int(row['late_papers'])}",
            va='center', fontsize=9)

fig.tight_layout()
plt.show()

# Also show new entrants (institutions that appeared only in the late period)
new_entrants = growth[
    (growth['early_papers'] == 0) & (growth['late_papers'] >= 3)
].sort_values('late_papers', ascending=False).head(10)

if len(new_entrants) > 0:
    print(f'\nNew entrants (no papers before {mid_year}, ≥3 after):')
    for _, row in new_entrants.iterrows():
        print(f"  {row['institution_name']}: {int(row['late_papers'])} papers")
else:
    print(f'\nNo new entrants with ≥3 papers found.')

---
## Save Results to DuckDB

In [ ]:
con.close()

con_write = duckdb.connect(DB)

# Institution paper counts
con_write.execute('CREATE OR REPLACE TABLE institution_summary AS SELECT * FROM ?',
                  [inst_paper_counts])

# Institution × topic counts (for dashboard)
inst_topic_counts = (
    inst_data
    .groupby(['institution_name', 'institution_country', 'topic_category'])
    ['openalex_id'].nunique()
    .reset_index(name='n_papers')
)
con_write.execute('CREATE OR REPLACE TABLE institution_topic_counts AS SELECT * FROM ?',
                  [inst_topic_counts])

# Collaboration density by topic
con_write.execute('CREATE OR REPLACE TABLE collaboration_density AS SELECT * FROM ?',
                  [collab_by_topic.reset_index()])

con_write.close()
print('Institutional landscape results saved to DuckDB.')